In [1]:
from selenium import webdriver
from bs4 import BeautifulSoup
import requests
import time
import json
import pandas as pd

# 爬取經濟日報即時新聞內容

In [2]:
news_section_urls =[ {'section': '要聞', 'url': 'https://money.udn.com/rank/newest/1001/10846/1'},
                   {'section': '產業', 'url': 'https://money.udn.com/rank/newest/1001/5591/1'},
                   {'section': '證券', 'url': 'https://money.udn.com/rank/newest/1001/5590/1'},
                   {'section': '國際', 'url': 'https://money.udn.com/rank/newest/1001/5588/1'},
                   {'section': '兩岸', 'url': 'https://money.udn.com/rank/newest/1001/5589/1'},
                   {'section': '金融', 'url': 'https://money.udn.com/rank/newest/1001/12017/1'},
                   {'section': '期貨', 'url': 'https://money.udn.com/rank/newest/1001/11111/1'},
                   {'section': '理財', 'url': 'https://money.udn.com/rank/newest/1001/5592/1'},
                   ]

In [3]:
def scrape_news_link(section, section_url):
    # Initialize Chrome webdriver
    driver = webdriver.Chrome()

    # Open the URL with Selenium
    driver.get(section_url)

    # Wait for some time to allow the page to load
    time.sleep(2)

    # Scroll down twice the height of the page」
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Wait for new content to load

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Wait for new content to load

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Wait for new content to load

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Wait for new content to load

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)  # Wait for new content to load

    # Get the updated HTML content after scrolling
    html_content = driver.page_source

    # Close the webdriver
    driver.quit()

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find all <div> tags under the specified class
    divs = soup.find_all('div', class_='story__content')

    # Extract href attribute from each <a> tag and text content from each <time> tag
    link_data = []
    for div in divs:
        link = div.a
        time_tag = div.find('time')
        if link and time_tag:
            href_link = link['href']
            time_content = time_tag.get_text()
            link_data.append({'news': '經濟日報', 'section': section,'link': href_link, 'time': time_content})

    print(link_data)
    return link_data

link_data =[]
for data in news_section_urls:
    links = scrape_news_link(data['section'], data['url'])
    link_data.append(links)
    print(len(links))

# Flatten the list using list comprehension
flatten_link_data = [item for sublist in link_data for item in sublist]
print(len(flatten_link_data))

[{'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/7307/7989507?from=edn_newestlist_rank', 'time': '2024-05-26 18:00'}, {'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/7307/7989506?from=edn_newestlist_rank', 'time': '2024-05-26 18:00'}, {'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/7307/7989474?from=edn_newestlist_rank', 'time': '2024-05-26 17:36'}, {'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/7307/7989415?from=edn_newestlist_rank', 'time': '2024-05-26 16:36'}, {'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/7307/7989413?from=edn_newestlist_rank', 'time': '2024-05-26 16:36'}, {'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/122335/7989408?from=edn_newestlist_rank', 'time': '2024-05-26 16:34'}, {'news': '經濟日報', 'section': '要聞', 'link': 'https://money.udn.com/money/story/122328/7989339?from=edn_newestlist_rank', 't

In [4]:
flatten_link_data

[{'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/story/7307/7989507?from=edn_newestlist_rank',
  'time': '2024-05-26 18:00'},
 {'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/story/7307/7989506?from=edn_newestlist_rank',
  'time': '2024-05-26 18:00'},
 {'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/story/7307/7989474?from=edn_newestlist_rank',
  'time': '2024-05-26 17:36'},
 {'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/story/7307/7989415?from=edn_newestlist_rank',
  'time': '2024-05-26 16:36'},
 {'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/story/7307/7989413?from=edn_newestlist_rank',
  'time': '2024-05-26 16:36'},
 {'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/story/122335/7989408?from=edn_newestlist_rank',
  'time': '2024-05-26 16:34'},
 {'news': '經濟日報',
  'section': '要聞',
  'link': 'https://money.udn.com/money/stor

In [5]:
# ===從連結抓新聞內容===

# get contents of the article
def get_content(url):
    # Send a GET request to the URL and fetch the HTML content
    response = requests.get(url)
    html_content = response.content

    # Parse the HTML content using Beautiful Soup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find title
    try:
        article_title = soup.find('h1', class_ = 'article-head__title').get_text(strip=True)
    except:
        article_title = None

    # Find source
    try:
        article_source = soup.find('div', class_ = 'article-body__info').get_text(strip=True)
    except:
        article_source = None

    # Find keywords
    try:
        article_keywords = soup.find('section', class_='article-keyword').get_text().split() ########
    except:
        article_keywords = None
    
    # Find text content
    try:
        article_content = soup.find('section', class_='article-body__editor').get_text()
    except:
        article_content = None
    try:
        article_content = article_content.split('延伸閱讀')[0]
    except:
        pass

    return article_title, article_source, article_keywords, article_content

# Iterate through the links
for data in flatten_link_data:
    link = data['link']
    title, source, keywords, content = get_content(link)
    print(title)
    data['title'], data['source'], data['keywords'], data['content'] = title, source, keywords, content



2波鋒面接力全台一週有雨 28日西半部防豪雨
柯建銘：全力阻擋藍白國會職權修法版本 絕對提釋憲
柯建銘批寡廉鮮恥 黃國昌：民進黨背棄承諾令人遺憾
變裝皇后妮妃雅關心政治 路透評：台灣非官方大使
觀宏專案疫後脫團率萬分之9  觀光署：沒有較高
經濟周報／美通膨關鍵數據、台股東會密集召開 台股5月最後一周怎麼走？
蛞蝓之亂從何來？攤商曝天候影響 還遇過菜裡有蛇「其實沒蟲才可怕」
阿里山林鐵觀光列車缺乏無障礙設施 身障團體促改善
自然碳匯有望新增 竹林、森林經營方法學審查中
陸軍：敵情威脅嚴峻 歸仁航特基地無法遷建
中國環台軍演 菲參議員：台灣在菲律賓有盟友
數位部三箭策略首推打詐  3個月內建置通報查詢網
馬辣「愛文芒果吃到飽」！還能升級東石大蛤蜊無限吃
馬辦：何時稱「我們中國人」是需要被原諒的事
空軍披露狙擊手莢艙標定畫面  學者推測共機渾然不知
民進黨縣市主委選舉投票  蘇巧慧：讓新北更進步
同框盧秀燕 柯文哲：民眾集結立院外有時空錯亂感
網友發起「青鳥行動」新竹場  籲5/28上街頭舉牌
部落文化保存者江萬花辭世  文化部將頒贈旌揚狀
高雄女孩籲不應立惡法 陳其邁︰提醒從政者傾聽民意
二膽島遭無人機投擲政治口號紙盒  軍方：認知戰伎倆
綠籲朝野坐下談 藍委建議總統邀在野黨領袖國政諮詢
藝人在中國被迫表態 賴總統：重要的是他們內心想法
拚民進黨高市主委 陳慧文、黃文益投完票握手展團結
吳思瑤：賴總統對失序國會正面喊話 在野黨充耳不聞
影／五月天自認中國人風波 賴清德表示不捨與諒解
迪士尼「小小世界」主題曲作者理查薛曼95歲去世
五月天、蔡依林疑被迫表態喊「我們中國」 柯文哲：無助兩岸交流
立院大戰國會改革法案 賴清德喊話：不可違法表決
美重申護台海和平 外交部：中國藉就職軍演 不會獲認同
陸環台軍演致兩岸緊張 賴清德喊話：理解、諒解、和解邁向和平共榮
外交部：理念盟友要求中國停止脅迫台灣破壞台海和平
空少飛遍70餘國學會泰語 疫情衝擊轉身變移民官
未來1周溼答答 周二鋒面到全台慎防豪雨 首颱北上恐掀長浪
提升存活率關鍵  國軍戰傷救護重要性一次看
推交通安全教育  教部補助建置教學材料包和影片
飄雨上看37度高溫 吳德榮：周二鋒面挾強對流 艾維尼颱風明生成大迴轉
鋒面要來了！ 慎防大雷雨突襲 周三雨勢驚人這裡恐紫爆
海峽雲雨帶移入中 南部3縣市不定時降雨
未來1周2

In [6]:
len(flatten_link_data)

2249

In [7]:
with open('經濟日報0526.json', 'w') as file:
    json.dump(flatten_link_data, file)

In [8]:
df = pd.read_json('經濟日報0526.json')
df.to_excel('經濟日報0526.xlsx', index = False)

In [40]:
link_data[100]

{'link': 'https://money.udn.com/money/story/7307/7917120?from=edn_newestlist_rank',
 'time': '2024-04-23 10:36',
 'title': '陳駿季基層研究員出身  化解進口蛋爭議獲肯定',
 'keywords': '\n\n\n農業部\n\n\n農委會\n\n\n屏東\n\n\n',
 'content': '\n內定農業部長由代理部長陳駿季真除，黨政人士今天說，陳駿季無鮮明黨派色彩，出身基層，用理性、專業解決政治問題，善盡政策說明責任，並在進口蛋爭議時邀集在野黨立委共同赴屏東監督銷毀作業，獲得肯定。\n總統當選人賴清德5月20日將就職，新內閣人事陸續出爐，內定行政院長卓榮泰今天上午公布第5波名單。\n\n\n\n\n\n\n\n\n\n黨政人士今天說，陳駿季出身農業部前身農委會，並在農業試驗所擔任所長9年的時間，自2019年接任農委會副主委，並在升格農業部後擔任政務次長。\n他說，陳駿季長年深耕農業科技研究領域，具備農糧研發的高度專業背景，並有多項專利註冊。研發人員出身的陳駿季曾表示，能讓研發技術落實、推廣到農業、政策上相當重要。在擔任農委會及農業部職位期間，陳駿季積極爭取科技研發預算，近年相關預算皆由陳駿季向國科會爭取成功，並獲得佳績。\n黨政人士表示，陳駿季在擔任代理部長期間，身段柔軟，對內延續政策，對國會積極說明，以高度專業獲得朝野立委的認可，並順利解決進口蛋爭議所引發的政治攻擊及口水。\n黨政人士指出，陳駿季無鮮明黨派色彩，出身基層的陳駿季用理性、專業解決政治問題，善盡政策說明責任，並在進口蛋爭議時邀集在野黨立委共同赴屏東監督銷毀作業，獲得肯定。\n黨政人士說，在面對國際上諸多挑戰，如氣候變遷及國際情勢高度變化的情況下，陳駿季過去投入大量心力在推動強化「精準調控、穩定生產」政策上，協助台灣因應瞬息多變的國際局勢，在2021年底順利育成44個新品種，超出原預計的30個。 ',
 'source': '中央社 記者葉素萍、賴于榛台北23日電'}